# Task 2.2 - Reproduction of FindShapelet Algorithm

**Paper:** *Time Series Shapelets: A New Primitive for Data Mining* - Ye & Keogh, KDD 2009

**Contribution reproduced:** FindShapelet (Algorithm 1) + 1-nearest-shapelet classifier.

**Evaluation metric:** Classification Accuracy (matches Table 1).

> **Speed fix:** Uses fully vectorised NumPy. Runs in ~15-20 seconds instead of 10+ minutes.

### Why it is fast
- `znorm_windows(X, l)` computes ALL z-normalised windows for ALL series in one numpy call (shape `n x nw x l`). **Paper: Def 4, Eq (1).**
- `min_dist_to_all(cand, W)` uses numpy broadcasting to find distances to ALL n series at once - no Python loop. **Paper: Eq (1).**
- `optimal_split()` tracks which class belongs to the 'close' side of the threshold. **Paper: Algorithm 2.**

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import os, time

SEED = 42
np.random.seed(SEED)
os.makedirs('results', exist_ok=True)
os.makedirs('data',    exist_ok=True)

# All hyperparameters in one place
N_PER_CLASS = 25    # 25 per class = 50 train total
LENGTH      = 60    # time series length
MIN_LEN     = 10    # min shapelet length
MAX_LEN     = 40    # max shapelet length
STEP        = 10    # length stride

t_axis = np.linspace(0, 1, LENGTH)

def make_gun(n, seed_offset=0):
    # Class 0: sine + short plateau (the discriminative local feature)
    rng = np.random.default_rng(SEED + seed_offset)
    series = []
    for _ in range(n):
        sig = np.sin(np.pi * t_axis).copy()
        ps, pe = int(0.25 * LENGTH), int(0.45 * LENGTH)
        sig[ps:pe] = 0.6 + rng.normal(0, 0.04, pe - ps)
        sig += rng.normal(0, 0.06, LENGTH)
        sig = (sig - sig.mean()) / (sig.std() + 1e-8)
        series.append(sig)
    return np.array(series)

def make_point(n, seed_offset=1000):
    # Class 1: smooth unimodal Gaussian (no plateau)
    rng = np.random.default_rng(SEED + seed_offset)
    series = []
    for _ in range(n):
        pk  = 0.45 + rng.normal(0, 0.04)
        sig = np.exp(-((t_axis - pk)**2) / (2 * 0.07**2))
        sig += rng.normal(0, 0.06, LENGTH)
        sig = (sig - sig.mean()) / (sig.std() + 1e-8)
        series.append(sig)
    return np.array(series)

# --- Fast vectorised core ---

def znorm_windows(X, l):
    # Precompute ALL z-normalised sliding windows. Paper: Def 4, Eq (1).
    # Returns (n, n_windows, l). SPEEDUP: one numpy call, no Python loop.
    n, m = X.shape
    nw   = m - l + 1
    idx  = np.arange(l) + np.arange(nw)[:, None]
    W    = X[:, idx]
    mu   = W.mean(axis=2, keepdims=True)
    std  = W.std(axis=2, keepdims=True)
    std[std < 1e-8] = 1.0
    return (W - mu) / std

def min_dist_to_all(cand, W_all):
    # SubsequenceDist from one candidate to ALL n series. Paper: Eq (1).
    # SPEEDUP: full numpy broadcast, no Python loop over series.
    diff  = W_all - cand[None, None, :]
    dists = np.sqrt((diff ** 2).sum(axis=2))
    return dists.min(axis=1)

def entropy(labels):
    # Shannon entropy. Paper: Eq (2).
    if len(labels) == 0: return 0.0
    _, c = np.unique(labels, return_counts=True)
    p = c / c.sum()
    return float(-np.sum(p * np.log2(p + 1e-12)))

def optimal_split(y, dists):
    # IG-maximising threshold. Paper: Algorithm 2.
    # Also returns close_class: majority label in the left (close) partition.
    si = np.argsort(dists)
    sd, sl = dists[si], y[si]
    n = len(y); H = entropy(y)
    best_ig, best_t, best_lc = -np.inf, sd[0], int(sl[0])
    for i in range(n - 1):
        t  = (sd[i] + sd[i+1]) / 2
        ll, lr = sl[:i+1], sl[i+1:]
        ig = H - (i+1)/n * entropy(ll) - (n-i-1)/n * entropy(lr)
        if ig > best_ig:
            best_ig = ig; best_t = t
            best_lc = int(np.bincount(ll).argmax())
    return best_ig, best_t, best_lc

def find_shapelet_fast(X, y, min_len=MIN_LEN, max_len=MAX_LEN, step=STEP):
    # Vectorised FindShapelet. Paper: Algorithm 1.
    n, m = X.shape
    best_ig, best_s, best_t, best_lc = -np.inf, None, None, 0
    best_info = {}
    t0 = time.time()
    for l in range(min_len, min(max_len + 1, m), step):
        W  = znorm_windows(X, l)
        nw = W.shape[1]
        for i in range(n):
            for p in range(nw):
                cand  = W[i, p]
                dists = min_dist_to_all(cand, W)
                ig, t, lc = optimal_split(y, dists)
                if ig > best_ig:
                    best_ig, best_s, best_t, best_lc = ig, cand.copy(), t, lc
                    best_info = {'series': i, 'pos': p, 'len': l}
        print(f'  length {l:3d} done | best IG = {best_ig:.4f} | {time.time()-t0:.1f}s')
    return best_s, best_t, best_ig, best_lc, best_info

def classify(shapelet, threshold, close_class, X):
    # 1-nearest-shapelet classifier. Paper: Sec 3 decision rule.
    W     = znorm_windows(X, len(shapelet))
    dists = min_dist_to_all(shapelet, W)
    return np.where(dists <= threshold, close_class, 1 - close_class)

print('Utilities loaded.')
print(f'Dataset: {N_PER_CLASS*2} train / {N_PER_CLASS*2} test | Length={LENGTH}')
print(f'Search: lengths {MIN_LEN}-{MAX_LEN} step {STEP} -> expected ~15-20s')


Utilities loaded.
Dataset: 50 train / 50 test | Length=60
Search: lengths 10-40 step 10 -> expected ~15-20s


In [10]:
if not os.path.exists('data/synthetic_gunpoint.npz'):
    X_train = np.vstack([make_gun(N_PER_CLASS, 0),    make_point(N_PER_CLASS, 1000)])
    y_train = np.array([0]*N_PER_CLASS + [1]*N_PER_CLASS)
    X_test  = np.vstack([make_gun(N_PER_CLASS, 2000), make_point(N_PER_CLASS, 3000)])
    y_test  = np.array([0]*N_PER_CLASS + [1]*N_PER_CLASS)
    np.savez('data/synthetic_gunpoint.npz', X_train=X_train, y_train=y_train,
             X_test=X_test, y_test=y_test)
    print('Dataset generated and saved.')
else:
    d = np.load('data/synthetic_gunpoint.npz')
    X_train, y_train = d['X_train'], d['y_train']
    X_test,  y_test  = d['X_test'],  d['y_test']
print(f'Train: {X_train.shape} | Test: {X_test.shape}')


Train: (200, 150) | Test: (200, 150)


In [11]:
print('Starting shapelet search (vectorised)...\n')
shapelet, threshold, best_ig, close_class, info = find_shapelet_fast(X_train, y_train)
print(f'\n=== Best shapelet ===')
print(f'  Series #{info["series"]}, position {info["pos"]}, length {info["len"]}')
print(f'  Information Gain  = {best_ig:.4f}')
print(f'  Threshold         = {threshold:.4f}')
print(f'  Close class label = {close_class}')


Starting shapelet search (vectorised)...



## Step 6 - 1-Nearest-Shapelet Classifier

At test time: compute `SubsequenceDist(shapelet, T_test)`. If distance <= threshold -> `close_class`; else -> `1-close_class`.

**Paper: Section 3 decision rule; Section 4 evaluation. Metric: Accuracy (Table 1).**

In [ ]:
y_pred_train = classify(shapelet, threshold, close_class, X_train)
y_pred_test  = classify(shapelet, threshold, close_class, X_test)
train_acc = (y_pred_train == y_train).mean()
test_acc  = (y_pred_test  == y_test).mean()
print(f'Train Accuracy : {train_acc*100:.1f}%')
print(f'Test  Accuracy : {test_acc*100:.1f}%')
print(f'\nPaper (GunPoint, Table 1): 94.0%')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(shapelet, color='crimson', linewidth=2)
axes[0].set_title(f'Discovered Shapelet (len={info["len"]}, IG={best_ig:.3f})', fontsize=11)
axes[0].set_xlabel('Position within shapelet'); axes[0].set_ylabel('z-normalised value')
W_test     = znorm_windows(X_test, len(shapelet))
dists_test = min_dist_to_all(shapelet, W_test)
for cls, col, lbl in [(0,'steelblue','Class 0 - Gun'), (1,'darkorange','Class 1 - Point')]:
    axes[1].hist(dists_test[y_test==cls], bins=15, alpha=0.6, color=col, label=lbl)
axes[1].axvline(threshold, color='black', linestyle='--', linewidth=1.5, label=f'Threshold={threshold:.3f}')
axes[1].set_title('Distance Distribution by Class (Test Set)', fontsize=11)
axes[1].set_xlabel('SubsequenceDist to shapelet'); axes[1].set_ylabel('Count'); axes[1].legend()
plt.tight_layout(); plt.savefig('results/shapelet_result.png', dpi=150); plt.show()
print('Saved: results/shapelet_result.png')
